# ShadowReliability: Final Submission Notebook

This notebook contains the final project workflow used for the submission version of ShadowReliability.

It starts from the final dataset setup and keeps the baseline, shadow model, corrected rich shadow, evaluation, and artifact saving sections that support the final report and app. Older pilot sections and trust feature side experiments were removed so the notebook is cleaner and easier to run from top to bottom.


## Full Baseline: TF-IDF and Logistic Regression

Imports and load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# 1) Load full master dataset
# -----------------------------
csv_path = Path("../data/healthcare_triage_master_900.csv")
df = pd.read_csv(csv_path)

print("Full dataset shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

print("\nOOD flag counts:")
print(df["ood_flag"].value_counts())



Full dataset shape: (900, 11)

Split counts:
split
train       360
val         180
test_id     180
test_ood    180
Name: count, dtype: int64

OOD flag counts:
ood_flag
0    720
1    180
Name: count, dtype: int64


Split dataframes

In [2]:
# -----------------------------
# 2) Split datasets
# -----------------------------
train_df = df[(df["split"] == "train") & (df["ood_flag"] == 0)].copy()
val_df = df[(df["split"] == "val") & (df["ood_flag"] == 0)].copy()
test_id_df = df[(df["split"] == "test_id") & (df["ood_flag"] == 0)].copy()
test_ood_df = df[df["split"] == "test_ood"].copy()

print("\nTrain shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test ID shape:", test_id_df.shape)
print("Test OOD shape:", test_ood_df.shape)

print("\nTrain label counts:")
print(train_df["label"].value_counts())

print("\nVal label counts:")
print(val_df["label"].value_counts())

print("\nTest ID label counts:")
print(test_id_df["label"].value_counts())


Train shape: (360, 11)
Val shape: (180, 11)
Test ID shape: (180, 11)
Test OOD shape: (180, 11)

Train label counts:
label
Appointment Scheduling       60
Billing and Insurance        60
Prescription Refill          60
Referral Request             60
Medical Records and Forms    60
Portal and Account Access    60
Name: count, dtype: int64

Val label counts:
label
Appointment Scheduling       30
Billing and Insurance        30
Prescription Refill          30
Referral Request             30
Medical Records and Forms    30
Portal and Account Access    30
Name: count, dtype: int64

Test ID label counts:
label
Appointment Scheduling       30
Billing and Insurance        30
Prescription Refill          30
Referral Request             30
Medical Records and Forms    30
Portal and Account Access    30
Name: count, dtype: int64


Build model

In [3]:
# -----------------------------
# 3) Build baseline model
# -----------------------------
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])


Train model

In [4]:
model.fit(train_df["text"], train_df["label"])

labels_sorted = sorted(train_df["label"].unique())


Validation results

In [5]:
# -----------------------------
# 5) Validation evaluation
# -----------------------------
val_pred = model.predict(val_df["text"])
val_acc = accuracy_score(val_df["label"], val_pred)

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"Validation accuracy: {val_acc:.4f}")

print("\nValidation classification report:")
print(classification_report(val_df["label"], val_pred, digits=3))

val_cm = confusion_matrix(val_df["label"], val_pred, labels=labels_sorted)
val_cm_df = pd.DataFrame(val_cm, index=labels_sorted, columns=labels_sorted)

print("\nValidation confusion matrix:")
print(val_cm_df)



VALIDATION RESULTS
Validation accuracy: 0.9944

Validation classification report:
                           precision    recall  f1-score   support

   Appointment Scheduling      1.000     0.967     0.983        30
    Billing and Insurance      1.000     1.000     1.000        30
Medical Records and Forms      1.000     1.000     1.000        30
Portal and Account Access      0.968     1.000     0.984        30
      Prescription Refill      1.000     1.000     1.000        30
         Referral Request      1.000     1.000     1.000        30

                 accuracy                          0.994       180
                macro avg      0.995     0.994     0.994       180
             weighted avg      0.995     0.994     0.994       180


Validation confusion matrix:
                           Appointment Scheduling  Billing and Insurance  \
Appointment Scheduling                         29                      0   
Billing and Insurance                           0             

Test ID results

In [6]:
# -----------------------------
# 6) Test ID evaluation
# -----------------------------
test_id_pred = model.predict(test_id_df["text"])
test_id_acc = accuracy_score(test_id_df["label"], test_id_pred)

print("\n" + "="*60)
print("TEST_ID RESULTS")
print("="*60)
print(f"Test ID accuracy: {test_id_acc:.4f}")

print("\nTest ID classification report:")
print(classification_report(test_id_df["label"], test_id_pred, digits=3))

test_id_cm = confusion_matrix(test_id_df["label"], test_id_pred, labels=labels_sorted)
test_id_cm_df = pd.DataFrame(test_id_cm, index=labels_sorted, columns=labels_sorted)

print("\nTest ID confusion matrix:")
print(test_id_cm_df)


TEST_ID RESULTS
Test ID accuracy: 1.0000

Test ID classification report:
                           precision    recall  f1-score   support

   Appointment Scheduling      1.000     1.000     1.000        30
    Billing and Insurance      1.000     1.000     1.000        30
Medical Records and Forms      1.000     1.000     1.000        30
Portal and Account Access      1.000     1.000     1.000        30
      Prescription Refill      1.000     1.000     1.000        30
         Referral Request      1.000     1.000     1.000        30

                 accuracy                          1.000       180
                macro avg      1.000     1.000     1.000       180
             weighted avg      1.000     1.000     1.000       180


Test ID confusion matrix:
                           Appointment Scheduling  Billing and Insurance  \
Appointment Scheduling                         30                      0   
Billing and Insurance                           0                     30  

Easy vs hard breakdown

In [7]:
# -----------------------------
# 7) Easy vs Hard breakdown
# -----------------------------
def split_accuracy(sub_df, name):
    pred = model.predict(sub_df["text"])
    acc = accuracy_score(sub_df["label"], pred)
    print(f"{name}: {acc:.4f}")

print("\n" + "="*60)
print("DIFFICULTY BREAKDOWN")
print("="*60)

split_accuracy(val_df[val_df["difficulty"] == "easy"], "Val easy accuracy")
split_accuracy(val_df[val_df["difficulty"] == "hard"], "Val hard accuracy")
split_accuracy(test_id_df[test_id_df["difficulty"] == "easy"], "Test ID easy accuracy")
split_accuracy(test_id_df[test_id_df["difficulty"] == "hard"], "Test ID hard accuracy")


DIFFICULTY BREAKDOWN
Val easy accuracy: 0.9921
Val hard accuracy: 1.0000
Test ID easy accuracy: 1.0000
Test ID hard accuracy: 1.0000


OOD confidence check

In [8]:
# -----------------------------
# 8) Confidence on ID vs OOD
# -----------------------------
val_probs = model.predict_proba(val_df["text"])
test_id_probs = model.predict_proba(test_id_df["text"])
ood_probs = model.predict_proba(test_ood_df["text"])

val_max_conf = val_probs.max(axis=1)
test_id_max_conf = test_id_probs.max(axis=1)
ood_max_conf = ood_probs.max(axis=1)

print("\n" + "="*60)
print("CONFIDENCE SUMMARY")
print("="*60)
print(f"Val mean max confidence:     {val_max_conf.mean():.4f}")
print(f"Test ID mean max confidence: {test_id_max_conf.mean():.4f}")
print(f"Test OOD mean max confidence:{ood_max_conf.mean():.4f}")

print("\nConfidence percentiles")
for name, arr in [
    ("Val", val_max_conf),
    ("Test ID", test_id_max_conf),
    ("Test OOD", ood_max_conf),
]:
    print(f"\n{name}:")
    print(pd.Series(arr).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))



CONFIDENCE SUMMARY
Val mean max confidence:     0.4711
Test ID mean max confidence: 0.4711
Test OOD mean max confidence:0.2373

Confidence percentiles

Val:
count    180.000000
mean       0.471084
std        0.099585
min        0.220295
10%        0.335372
25%        0.402746
50%        0.472491
75%        0.543016
90%        0.589048
max        0.750709
dtype: float64

Test ID:
count    180.000000
mean       0.471087
std        0.099993
min        0.215000
10%        0.350301
25%        0.403339
50%        0.466190
75%        0.543656
90%        0.589048
max        0.808078
dtype: float64

Test OOD:
count    180.000000
mean       0.237274
std        0.043025
min        0.176487
10%        0.196000
25%        0.207150
50%        0.224251
75%        0.253765
90%        0.288310
max        0.423324
dtype: float64


OOD confidence check

In [9]:
# -----------------------------
# 9) Save quick prediction tables (optional but useful)
# -----------------------------
val_out = val_df[["text", "label", "difficulty"]].copy()
val_out["pred"] = val_pred
val_out["correct"] = (val_out["label"] == val_out["pred"]).astype(int)
val_out["max_conf"] = val_max_conf

test_id_out = test_id_df[["text", "label", "difficulty"]].copy()
test_id_out["pred"] = test_id_pred
test_id_out["correct"] = (test_id_out["label"] == test_id_out["pred"]).astype(int)
test_id_out["max_conf"] = test_id_max_conf

ood_out = test_ood_df[["text", "ood_subtype"]].copy()
ood_out["pred"] = model.predict(test_ood_df["text"])
ood_out["max_conf"] = ood_max_conf

val_out.to_csv("baseline_val_predictions.csv", index=False)
test_id_out.to_csv("baseline_test_id_predictions.csv", index=False)
ood_out.to_csv("baseline_test_ood_predictions.csv", index=False)

print("\nSaved:")
print("- baseline_val_predictions.csv")
print("- baseline_test_id_predictions.csv")
print("- baseline_test_ood_predictions.csv")


Saved:
- baseline_val_predictions.csv
- baseline_test_id_predictions.csv
- baseline_test_ood_predictions.csv


## Full Baseline Re-run with Challenge Set

Load base dataset and challenge dataset

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

base_path = Path("../data/healthcare_triage_master_900.csv")
challenge_path = Path("../data/healthcare_triage_test_challenge_72_merged.csv")

df_base = pd.read_csv(base_path)
df_challenge = pd.read_csv(challenge_path)

print("Base shape:", df_base.shape)
print("Challenge shape:", df_challenge.shape)

# combine
df = pd.concat([df_base, df_challenge], ignore_index=True)

print("\nCombined shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

Base shape: (900, 11)
Challenge shape: (72, 11)

Combined shape: (972, 11)

Split counts:
split
train             360
val               180
test_id           180
test_ood          180
test_challenge     72
Name: count, dtype: int64


Split datasets

In [11]:
train_df = df[(df["split"] == "train") & (df["ood_flag"] == 0)].copy()
val_df = df[(df["split"] == "val") & (df["ood_flag"] == 0)].copy()
test_id_df = df[(df["split"] == "test_id") & (df["ood_flag"] == 0)].copy()
test_challenge_df = df[(df["split"] == "test_challenge") & (df["ood_flag"] == 0)].copy()
test_ood_df = df[df["split"] == "test_ood"].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test ID:", test_id_df.shape)
print("Test Challenge:", test_challenge_df.shape)
print("Test OOD:", test_ood_df.shape)

print("\nChallenge label counts:")
print(test_challenge_df["label"].value_counts())

Train: (360, 11)
Val: (180, 11)
Test ID: (180, 11)
Test Challenge: (72, 11)
Test OOD: (180, 11)

Challenge label counts:
label
Appointment Scheduling       12
Referral Request             12
Medical Records and Forms    12
Portal and Account Access    12
Billing and Insurance        12
Prescription Refill          12
Name: count, dtype: int64


Build and train the same baseline model

In [12]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(train_df["text"], train_df["label"])

labels_sorted = sorted(train_df["label"].unique())

Validation results

In [13]:
val_pred = model.predict(val_df["text"])
val_acc = accuracy_score(val_df["label"], val_pred)

print("=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)
print(f"Validation accuracy: {val_acc:.4f}\n")

print(classification_report(val_df["label"], val_pred, digits=3))

val_cm = confusion_matrix(val_df["label"], val_pred, labels=labels_sorted)
val_cm_df = pd.DataFrame(val_cm, index=labels_sorted, columns=labels_sorted)

print("\nValidation confusion matrix:")
print(val_cm_df)

VALIDATION RESULTS
Validation accuracy: 0.9944

                           precision    recall  f1-score   support

   Appointment Scheduling      1.000     0.967     0.983        30
    Billing and Insurance      1.000     1.000     1.000        30
Medical Records and Forms      1.000     1.000     1.000        30
Portal and Account Access      0.968     1.000     0.984        30
      Prescription Refill      1.000     1.000     1.000        30
         Referral Request      1.000     1.000     1.000        30

                 accuracy                          0.994       180
                macro avg      0.995     0.994     0.994       180
             weighted avg      0.995     0.994     0.994       180


Validation confusion matrix:
                           Appointment Scheduling  Billing and Insurance  \
Appointment Scheduling                         29                      0   
Billing and Insurance                           0                     30   
Medical Records and F

Test ID results

In [14]:
test_id_pred = model.predict(test_id_df["text"])
test_id_acc = accuracy_score(test_id_df["label"], test_id_pred)

print("=" * 60)
print("TEST_ID RESULTS")
print("=" * 60)
print(f"Test ID accuracy: {test_id_acc:.4f}\n")

print(classification_report(test_id_df["label"], test_id_pred, digits=3))

test_id_cm = confusion_matrix(test_id_df["label"], test_id_pred, labels=labels_sorted)
test_id_cm_df = pd.DataFrame(test_id_cm, index=labels_sorted, columns=labels_sorted)

print("\nTest ID confusion matrix:")
print(test_id_cm_df)

TEST_ID RESULTS
Test ID accuracy: 1.0000

                           precision    recall  f1-score   support

   Appointment Scheduling      1.000     1.000     1.000        30
    Billing and Insurance      1.000     1.000     1.000        30
Medical Records and Forms      1.000     1.000     1.000        30
Portal and Account Access      1.000     1.000     1.000        30
      Prescription Refill      1.000     1.000     1.000        30
         Referral Request      1.000     1.000     1.000        30

                 accuracy                          1.000       180
                macro avg      1.000     1.000     1.000       180
             weighted avg      1.000     1.000     1.000       180


Test ID confusion matrix:
                           Appointment Scheduling  Billing and Insurance  \
Appointment Scheduling                         30                      0   
Billing and Insurance                           0                     30   
Medical Records and Forms     

Challenge set results

In [15]:
challenge_pred = model.predict(test_challenge_df["text"])
challenge_acc = accuracy_score(test_challenge_df["label"], challenge_pred)

print("=" * 60)
print("TEST_CHALLENGE RESULTS")
print("=" * 60)
print(f"Test Challenge accuracy: {challenge_acc:.4f}\n")

print(classification_report(test_challenge_df["label"], challenge_pred, digits=3))

challenge_cm = confusion_matrix(test_challenge_df["label"], challenge_pred, labels=labels_sorted)
challenge_cm_df = pd.DataFrame(challenge_cm, index=labels_sorted, columns=labels_sorted)

print("\nTest Challenge confusion matrix:")
print(challenge_cm_df)

TEST_CHALLENGE RESULTS
Test Challenge accuracy: 0.8611

                           precision    recall  f1-score   support

   Appointment Scheduling      0.909     0.833     0.870        12
    Billing and Insurance      0.647     0.917     0.759        12
Medical Records and Forms      1.000     0.667     0.800        12
Portal and Account Access      0.909     0.833     0.870        12
      Prescription Refill      0.917     0.917     0.917        12
         Referral Request      0.923     1.000     0.960        12

                 accuracy                          0.861        72
                macro avg      0.884     0.861     0.862        72
             weighted avg      0.884     0.861     0.862        72


Test Challenge confusion matrix:
                           Appointment Scheduling  Billing and Insurance  \
Appointment Scheduling                         10                      1   
Billing and Insurance                           0                     11   
Medical R

Easy / hard / challenge comparison

In [16]:
def eval_split(name, sub_df):
    pred = model.predict(sub_df["text"])
    acc = accuracy_score(sub_df["label"], pred)
    print(f"{name}: {acc:.4f}")

print("=" * 60)
print("DIFFICULTY / SPLIT COMPARISON")
print("=" * 60)

eval_split("Val easy", val_df[val_df["difficulty"] == "easy"])
eval_split("Val hard", val_df[val_df["difficulty"] == "hard"])

eval_split("Test ID easy", test_id_df[test_id_df["difficulty"] == "easy"])
eval_split("Test ID hard", test_id_df[test_id_df["difficulty"] == "hard"])

eval_split("Test Challenge", test_challenge_df)

DIFFICULTY / SPLIT COMPARISON
Val easy: 0.9921
Val hard: 1.0000
Test ID easy: 1.0000
Test ID hard: 1.0000
Test Challenge: 0.8611


Confidence comparison: val vs test_id vs challenge vs OOD

In [17]:
val_probs = model.predict_proba(val_df["text"])
test_id_probs = model.predict_proba(test_id_df["text"])
challenge_probs = model.predict_proba(test_challenge_df["text"])
ood_probs = model.predict_proba(test_ood_df["text"])

val_max_conf = val_probs.max(axis=1)
test_id_max_conf = test_id_probs.max(axis=1)
challenge_max_conf = challenge_probs.max(axis=1)
ood_max_conf = ood_probs.max(axis=1)

print("=" * 60)
print("CONFIDENCE SUMMARY")
print("=" * 60)
print(f"Val mean max confidence:          {val_max_conf.mean():.4f}")
print(f"Test ID mean max confidence:      {test_id_max_conf.mean():.4f}")
print(f"Test Challenge mean max confidence:{challenge_max_conf.mean():.4f}")
print(f"Test OOD mean max confidence:     {ood_max_conf.mean():.4f}")

for name, arr in [
    ("Val", val_max_conf),
    ("Test ID", test_id_max_conf),
    ("Test Challenge", challenge_max_conf),
    ("Test OOD", ood_max_conf),
]:
    print(f"\n{name}:")
    print(pd.Series(arr).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

CONFIDENCE SUMMARY
Val mean max confidence:          0.4711
Test ID mean max confidence:      0.4711
Test Challenge mean max confidence:0.3000
Test OOD mean max confidence:     0.2373

Val:
count    180.000000
mean       0.471084
std        0.099585
min        0.220295
10%        0.335372
25%        0.402746
50%        0.472491
75%        0.543016
90%        0.589048
max        0.750709
dtype: float64

Test ID:
count    180.000000
mean       0.471087
std        0.099993
min        0.215000
10%        0.350301
25%        0.403339
50%        0.466190
75%        0.543656
90%        0.589048
max        0.808078
dtype: float64

Test Challenge:
count    72.000000
mean      0.299988
std       0.069874
min       0.200675
10%       0.229482
25%       0.253297
50%       0.284414
75%       0.329809
90%       0.415830
max       0.546813
dtype: float64

Test OOD:
count    180.000000
mean       0.237274
std        0.043025
min        0.176487
10%        0.196000
25%        0.207150
50%        0.2242

Save prediction files

In [18]:
val_out = val_df[["text", "label", "difficulty"]].copy()
val_out["pred"] = val_pred
val_out["correct"] = (val_out["label"] == val_out["pred"]).astype(int)
val_out["max_conf"] = val_max_conf

test_id_out = test_id_df[["text", "label", "difficulty"]].copy()
test_id_out["pred"] = test_id_pred
test_id_out["correct"] = (test_id_out["label"] == test_id_out["pred"]).astype(int)
test_id_out["max_conf"] = test_id_max_conf

challenge_out = test_challenge_df[["text", "label", "difficulty"]].copy()
challenge_out["pred"] = challenge_pred
challenge_out["correct"] = (challenge_out["label"] == challenge_out["pred"]).astype(int)
challenge_out["max_conf"] = challenge_max_conf

ood_out = test_ood_df[["text", "ood_subtype"]].copy()
ood_out["pred"] = model.predict(test_ood_df["text"])
ood_out["max_conf"] = ood_max_conf

val_out.to_csv("baseline_val_predictions_v2.csv", index=False)
test_id_out.to_csv("baseline_test_id_predictions_v2.csv", index=False)
challenge_out.to_csv("baseline_test_challenge_predictions.csv", index=False)
ood_out.to_csv("baseline_test_ood_predictions_v2.csv", index=False)

print("Saved:")
print("- baseline_val_predictions_v2.csv")
print("- baseline_test_id_predictions_v2.csv")
print("- baseline_test_challenge_predictions.csv")
print("- baseline_test_ood_predictions_v2.csv")

Saved:
- baseline_val_predictions_v2.csv
- baseline_test_id_predictions_v2.csv
- baseline_test_challenge_predictions.csv
- baseline_test_ood_predictions_v2.csv


## calibration and confidence-threshold selective prediction baseline

imports

In [19]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

train raw model and calibrated model

In [20]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Rebuild raw base pipeline
raw_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

# Fit raw model on train
raw_model.fit(train_df["text"], train_df["label"])

# Calibrate using validation set
cal_model = CalibratedClassifierCV(
    FrozenEstimator(raw_model),
    method="sigmoid"
)
cal_model.fit(val_df["text"], val_df["label"])

print("Raw model and calibrated model are ready.")

Raw model and calibrated model are ready.


quick calibration oriented metrics

In [21]:
def multiclass_brier_score(y_true, probas, class_order):
    y_onehot = pd.get_dummies(pd.Categorical(y_true, categories=class_order)).values
    return np.mean(np.sum((probas - y_onehot) ** 2, axis=1))

# Validation probabilities
val_raw_probs = raw_model.predict_proba(val_df["text"])
val_cal_probs = cal_model.predict_proba(val_df["text"])

val_raw_pred = raw_model.predict(val_df["text"])
val_cal_pred = cal_model.predict(val_df["text"])

print("Validation accuracy (raw):", accuracy_score(val_df["label"], val_raw_pred))
print("Validation accuracy (cal):", accuracy_score(val_df["label"], val_cal_pred))

print("\nValidation log loss (raw):", log_loss(val_df["label"], val_raw_probs, labels=labels_sorted))
print("Validation log loss (cal):", log_loss(val_df["label"], val_cal_probs, labels=labels_sorted))

print("\nValidation multiclass Brier score (raw):", multiclass_brier_score(val_df["label"], val_raw_probs, labels_sorted))
print("Validation multiclass Brier score (cal):", multiclass_brier_score(val_df["label"], val_cal_probs, labels_sorted))

Validation accuracy (raw): 0.9944444444444445
Validation accuracy (cal): 1.0

Validation log loss (raw): 0.7764929417922787
Validation log loss (cal): 0.0733072428672358

Validation multiclass Brier score (raw): 0.3495161892067379
Validation multiclass Brier score (cal): 0.013557175423300493


confidence threshold selective prediction function

In [22]:
def selective_metrics(y_true, probas, threshold):
    pred = probas.argmax(axis=1)
    pred_labels = np.array(labels_sorted)[pred]
    max_conf = probas.max(axis=1)

    keep_mask = max_conf >= threshold
    coverage = keep_mask.mean()

    if keep_mask.sum() == 0:
        return {
            "threshold": threshold,
            "coverage": 0.0,
            "accuracy_on_kept": np.nan,
            "error_on_kept": np.nan,
            "num_kept": 0
        }

    kept_true = np.array(y_true)[keep_mask]
    kept_pred = pred_labels[keep_mask]
    acc = accuracy_score(kept_true, kept_pred)

    return {
        "threshold": threshold,
        "coverage": coverage,
        "accuracy_on_kept": acc,
        "error_on_kept": 1 - acc,
        "num_kept": int(keep_mask.sum())
    }

run threshold sweep on test_id and test_challenge

In [23]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

test_id_cal_probs = cal_model.predict_proba(test_id_df["text"])
challenge_cal_probs = cal_model.predict_proba(test_challenge_df["text"])

test_id_results = []
challenge_results = []

for t in thresholds:
    test_id_results.append(selective_metrics(test_id_df["label"], test_id_cal_probs, t))
    challenge_results.append(selective_metrics(test_challenge_df["label"], challenge_cal_probs, t))

test_id_results_df = pd.DataFrame(test_id_results)
challenge_results_df = pd.DataFrame(challenge_results)

print("TEST_ID selective prediction results:")
print(test_id_results_df)

print("\nTEST_CHALLENGE selective prediction results:")
print(challenge_results_df)

TEST_ID selective prediction results:
   threshold  coverage  accuracy_on_kept  error_on_kept  num_kept
0        0.2  1.000000               1.0            0.0       180
1        0.3  1.000000               1.0            0.0       180
2        0.4  1.000000               1.0            0.0       180
3        0.5  1.000000               1.0            0.0       180
4        0.6  0.983333               1.0            0.0       177
5        0.7  0.972222               1.0            0.0       175
6        0.8  0.955556               1.0            0.0       172

TEST_CHALLENGE selective prediction results:
   threshold  coverage  accuracy_on_kept  error_on_kept  num_kept
0        0.2  1.000000          0.861111       0.138889        72
1        0.3  1.000000          0.861111       0.138889        72
2        0.4  0.986111          0.873239       0.126761        71
3        0.5  0.916667          0.878788       0.121212        66
4        0.6  0.777778          0.910714       0.089286   

compare raw vs calibrated confidence means

In [24]:
test_id_raw_probs = raw_model.predict_proba(test_id_df["text"])
challenge_raw_probs = raw_model.predict_proba(test_challenge_df["text"])

print("Mean max confidence:")
print("\nRaw model:")
print("  Test ID:", test_id_raw_probs.max(axis=1).mean())
print("  Test Challenge:", challenge_raw_probs.max(axis=1).mean())

print("\nCalibrated model:")
print("  Test ID:", test_id_cal_probs.max(axis=1).mean())
print("  Test Challenge:", challenge_cal_probs.max(axis=1).mean())

Mean max confidence:

Raw model:
  Test ID: 0.4710868344002375
  Test Challenge: 0.29998815180043203

Calibrated model:
  Test ID: 0.9318753384752967
  Test Challenge: 0.7340976857732477


save threshold tables

In [25]:
test_id_results_df.to_csv("selective_test_id_results.csv", index=False)
challenge_results_df.to_csv("selective_test_challenge_results.csv", index=False)

print("Saved:")
print("- selective_test_id_results.csv")
print("- selective_test_challenge_results.csv")

Saved:
- selective_test_id_results.csv
- selective_test_challenge_results.csv


In [26]:
import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

def extract_shadow_features(probas, texts):
    """
    probas: np.ndarray of shape (n_samples, n_classes)
    texts: iterable of raw text strings
    """
    max_conf = probas.max(axis=1)

    sorted_probs = np.sort(probas, axis=1)
    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2]
    margin = top1 - top2

    ent = np.apply_along_axis(lambda x: scipy_entropy(x + 1e-12), 1, probas)

    text_len_chars = np.array([len(str(t)) for t in texts])
    text_len_words = np.array([len(str(t).split()) for t in texts])

    feat_df = pd.DataFrame({
        "max_conf": max_conf,
        "margin": margin,
        "entropy": ent,
        "text_len_chars": text_len_chars,
        "text_len_words": text_len_words,
    })

    return feat_df

# Shadow Model: Error Prediction

building out of fold shadow training data from train

In [27]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

# Use the same base pipeline as your raw model
base_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros((len(train_df), len(labels_sorted)))
oof_pred = np.empty(len(train_df), dtype=object)

X_train_all = train_df["text"].reset_index(drop=True)
y_train_all = train_df["label"].reset_index(drop=True)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_all, y_train_all), 1):
    fold_model = clone(base_pipeline)
    fold_model.fit(X_train_all.iloc[tr_idx], y_train_all.iloc[tr_idx])

    fold_probs = fold_model.predict_proba(X_train_all.iloc[va_idx])
    fold_preds = fold_model.predict(X_train_all.iloc[va_idx])

    oof_probs[va_idx] = fold_probs
    oof_pred[va_idx] = fold_preds

print("OOF predictions complete.")

OOF predictions complete.


make shadow training features/labels

In [28]:
# Error target from OOF predictions
shadow_train_y = (oof_pred != y_train_all.values).astype(int)

# Features from OOF probabilities + raw text
shadow_train_X = extract_shadow_features(oof_probs, X_train_all)

print("Shadow training feature shape:", shadow_train_X.shape)
print("\nShadow target distribution:")
print(pd.Series(shadow_train_y).value_counts())
print("\nShadow error rate on train OOF:", shadow_train_y.mean())

Shadow training feature shape: (360, 5)

Shadow target distribution:
0    339
1     21
Name: count, dtype: int64

Shadow error rate on train OOF: 0.058333333333333334


keep challenge as the shadow test set

In [29]:
# Main calibrated model predictions on challenge (same as before)
challenge_shadow_probs = cal_model.predict_proba(test_challenge_df["text"])
challenge_shadow_pred = cal_model.predict(test_challenge_df["text"])
challenge_error_label = (challenge_shadow_pred != test_challenge_df["label"]).astype(int)

shadow_test_X = extract_shadow_features(challenge_shadow_probs, test_challenge_df["text"])
shadow_test_y = challenge_error_label

print("Shadow challenge feature shape:", shadow_test_X.shape)
print("\nChallenge error target distribution:")
print(pd.Series(shadow_test_y).value_counts())
print("\nChallenge error rate:", shadow_test_y.mean())

Shadow challenge feature shape: (72, 5)

Challenge error target distribution:
label
0    62
1    10
Name: count, dtype: int64

Challenge error rate: 0.1388888888888889


train shadow model (Logistic Regression shadow model)

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

shadow_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

shadow_model.fit(shadow_train_X, shadow_train_y)

print("Shadow model trained.")

Shadow model trained.


Evaluate shadow model on challenge

In [31]:
shadow_test_risk = shadow_model.predict_proba(shadow_test_X)[:, 1]
shadow_test_pred = shadow_model.predict(shadow_test_X)

print("Shadow model positive prediction count:", shadow_test_pred.sum())

if len(np.unique(shadow_test_y)) == 2:
    roc_auc = roc_auc_score(shadow_test_y, shadow_test_risk)
    ap = average_precision_score(shadow_test_y, shadow_test_risk)

    print(f"Shadow ROC-AUC on challenge: {roc_auc:.4f}")
    print(f"Shadow Average Precision on challenge: {ap:.4f}")
else:
    print("Only one class present in challenge error labels; AUC/AP not defined.")

Shadow model positive prediction count: 0
Shadow ROC-AUC on challenge: 0.8129
Shadow Average Precision on challenge: 0.4803


Evaluate shadow model on challenge

In [32]:
shadow_test_risk = shadow_model.predict_proba(shadow_test_X)[:, 1]
shadow_test_pred = shadow_model.predict(shadow_test_X)

print("Shadow model positive prediction count:", shadow_test_pred.sum())

if len(np.unique(shadow_test_y)) == 2:
    roc_auc = roc_auc_score(shadow_test_y, shadow_test_risk)
    ap = average_precision_score(shadow_test_y, shadow_test_risk)

    print(f"Shadow ROC-AUC on challenge: {roc_auc:.4f}")
    print(f"Shadow Average Precision on challenge: {ap:.4f}")
else:
    print("Only one class present in challenge error labels; AUC/AP not defined.")

Shadow model positive prediction count: 0
Shadow ROC-AUC on challenge: 0.8129
Shadow Average Precision on challenge: 0.4803


Feature importance / coefficients

In [33]:
coef_df = pd.DataFrame({
    "feature": shadow_train_X.columns,
    "coef": shadow_model.coef_[0]
}).sort_values("coef", ascending=False)

print("Shadow model coefficients:")
print(coef_df)

Shadow model coefficients:
          feature      coef
2         entropy  2.383798
4  text_len_words  0.258874
3  text_len_chars -0.057880
0        max_conf -3.636117
1          margin -5.345520


Shadow based abstention table

In [34]:
def selective_metrics_from_risk(y_true, y_pred_labels, risk_scores, risk_threshold):
    keep_mask = risk_scores <= risk_threshold
    coverage = keep_mask.mean()

    if keep_mask.sum() == 0:
        return {
            "risk_threshold": risk_threshold,
            "coverage": 0.0,
            "accuracy_on_kept": np.nan,
            "error_on_kept": np.nan,
            "num_kept": 0
        }

    kept_true = np.array(y_true)[keep_mask]
    kept_pred = np.array(y_pred_labels)[keep_mask]
    acc = accuracy_score(kept_true, kept_pred)

    return {
        "risk_threshold": risk_threshold,
        "coverage": coverage,
        "accuracy_on_kept": acc,
        "error_on_kept": 1 - acc,
        "num_kept": int(keep_mask.sum())
    }

risk_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

shadow_results = []
for t in risk_thresholds:
    shadow_results.append(
        selective_metrics_from_risk(
            y_true=test_challenge_df["label"],
            y_pred_labels=challenge_shadow_pred,
            risk_scores=shadow_test_risk,
            risk_threshold=t
        )
    )

shadow_results_df = pd.DataFrame(shadow_results)

print("SHADOW selective prediction results on challenge:")
print(shadow_results_df)

SHADOW selective prediction results on challenge:
   risk_threshold  coverage  accuracy_on_kept  error_on_kept  num_kept
0             0.2  0.986111          0.873239       0.126761        71
1             0.3  0.986111          0.873239       0.126761        71
2             0.4  0.986111          0.873239       0.126761        71
3             0.5  1.000000          0.861111       0.138889        72
4             0.6  1.000000          0.861111       0.138889        72
5             0.7  1.000000          0.861111       0.138889        72
6             0.8  1.000000          0.861111       0.138889        72


Save outputs

In [35]:
challenge_shadow_out = test_challenge_df[["text", "label", "difficulty"]].copy()
challenge_shadow_out["main_pred"] = challenge_shadow_pred
challenge_shadow_out["main_correct"] = (challenge_shadow_out["label"] == challenge_shadow_out["main_pred"]).astype(int)
challenge_shadow_out["main_max_conf"] = challenge_shadow_probs.max(axis=1)
challenge_shadow_out["shadow_risk"] = shadow_test_risk

challenge_shadow_out.to_csv("shadow_test_challenge_predictions.csv", index=False)
shadow_results_df.to_csv("shadow_selective_results_challenge.csv", index=False)

print("Saved:")
print("- shadow_test_challenge_predictions.csv")
print("- shadow_selective_results_challenge.csv")

Saved:
- shadow_test_challenge_predictions.csv
- shadow_selective_results_challenge.csv


# Stronger Shadow Model Check (Random Forest)

train random forest shadow model

In [36]:
from sklearn.ensemble import RandomForestClassifier

rf_shadow_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

rf_shadow_model.fit(shadow_train_X, shadow_train_y)

print("Random forest shadow model trained.")

Random forest shadow model trained.


evaluate on challenge

In [37]:
rf_shadow_test_risk = rf_shadow_model.predict_proba(shadow_test_X)[:, 1]
rf_shadow_test_pred = rf_shadow_model.predict(shadow_test_X)

print("RF shadow model positive prediction count:", rf_shadow_test_pred.sum())

if len(np.unique(shadow_test_y)) == 2:
    rf_roc_auc = roc_auc_score(shadow_test_y, rf_shadow_test_risk)
    rf_ap = average_precision_score(shadow_test_y, rf_shadow_test_risk)

    print(f"RF Shadow ROC-AUC on challenge: {rf_roc_auc:.4f}")
    print(f"RF Shadow Average Precision on challenge: {rf_ap:.4f}")
else:
    print("Only one class present in challenge error labels; AUC/AP not defined.")

RF shadow model positive prediction count: 0
RF Shadow ROC-AUC on challenge: 0.5790
RF Shadow Average Precision on challenge: 0.2778


feature importance

In [38]:
rf_importance_df = pd.DataFrame({
    "feature": shadow_train_X.columns,
    "importance": rf_shadow_model.feature_importances_
}).sort_values("importance", ascending=False)

print("RF shadow feature importances:")
print(rf_importance_df)

RF shadow feature importances:
          feature  importance
1          margin    0.437011
0        max_conf    0.277828
2         entropy    0.171771
3  text_len_chars    0.059137
4  text_len_words    0.054253


RF shadow abstention table

In [39]:
rf_shadow_results = []
for t in risk_thresholds:
    rf_shadow_results.append(
        selective_metrics_from_risk(
            y_true=test_challenge_df["label"],
            y_pred_labels=challenge_shadow_pred,   # keep same main-model predictions
            risk_scores=rf_shadow_test_risk,
            risk_threshold=t
        )
    )

rf_shadow_results_df = pd.DataFrame(rf_shadow_results)

print("RF SHADOW selective prediction results on challenge:")
print(rf_shadow_results_df)

RF SHADOW selective prediction results on challenge:
   risk_threshold  coverage  accuracy_on_kept  error_on_kept  num_kept
0             0.2  0.972222          0.871429       0.128571        70
1             0.3  1.000000          0.861111       0.138889        72
2             0.4  1.000000          0.861111       0.138889        72
3             0.5  1.000000          0.861111       0.138889        72
4             0.6  1.000000          0.861111       0.138889        72
5             0.7  1.000000          0.861111       0.138889        72
6             0.8  1.000000          0.861111       0.138889        72


save outputs

In [40]:
rf_shadow_results_df.to_csv("rf_shadow_selective_results_challenge.csv", index=False)

rf_challenge_shadow_out = test_challenge_df[["text", "label", "difficulty"]].copy()
rf_challenge_shadow_out["main_pred"] = challenge_shadow_pred
rf_challenge_shadow_out["main_correct"] = (rf_challenge_shadow_out["label"] == rf_challenge_shadow_out["main_pred"]).astype(int)
rf_challenge_shadow_out["rf_shadow_risk"] = rf_shadow_test_risk

rf_challenge_shadow_out.to_csv("rf_shadow_test_challenge_predictions.csv", index=False)

print("Saved:")
print("- rf_shadow_selective_results_challenge.csv")
print("- rf_shadow_test_challenge_predictions.csv")

Saved:
- rf_shadow_selective_results_challenge.csv
- rf_shadow_test_challenge_predictions.csv


## Multi Model Baselines for ShadowReliability

imports for extra models

In [41]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

build the extra main models

In [42]:
# Logistic model already exists conceptually, but we rebuild cleanly here for consistency
lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

svm_base = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", MultinomialNB()
    )
])

print("Models defined.")

Models defined.


fit the models

In [43]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Fit Logistic Regression
lr_model.fit(train_df["text"], train_df["label"])

# Fit Naive Bayes
nb_model.fit(train_df["text"], train_df["label"])

# Fit SVM base model first
svm_base.fit(train_df["text"], train_df["label"])

# Calibrate SVM using validation set
svm_model = CalibratedClassifierCV(
    FrozenEstimator(svm_base),
    method="sigmoid"
)
svm_model.fit(val_df["text"], val_df["label"])

print("All three models trained.")

All three models trained.


evaluate all 3 on validation, test_id, challenge

In [44]:
def quick_eval(model, df_eval, name):
    pred = model.predict(df_eval["text"])
    acc = accuracy_score(df_eval["label"], pred)
    print(f"{name}: {acc:.4f}")

print("Validation")
quick_eval(lr_model, val_df, "LR val")
quick_eval(svm_model, val_df, "SVM val")
quick_eval(nb_model, val_df, "NB val")

print("\nTest ID")
quick_eval(lr_model, test_id_df, "LR test_id")
quick_eval(svm_model, test_id_df, "SVM test_id")
quick_eval(nb_model, test_id_df, "NB test_id")

print("\nTest Challenge")
quick_eval(lr_model, test_challenge_df, "LR challenge")
quick_eval(svm_model, test_challenge_df, "SVM challenge")
quick_eval(nb_model, test_challenge_df, "NB challenge")

Validation
LR val: 0.9944
SVM val: 1.0000
NB val: 0.9889

Test ID
LR test_id: 1.0000
SVM test_id: 1.0000
NB test_id: 0.9944

Test Challenge
LR challenge: 0.8611
SVM challenge: 0.8611
NB challenge: 0.8472


get predictions and probabilities for challenge

In [45]:
# Logistic
lr_challenge_pred = lr_model.predict(test_challenge_df["text"])
lr_challenge_probs = lr_model.predict_proba(test_challenge_df["text"])

# SVM calibrated
svm_challenge_pred = svm_model.predict(test_challenge_df["text"])
svm_challenge_probs = svm_model.predict_proba(test_challenge_df["text"])

# Naive Bayes
nb_challenge_pred = nb_model.predict(test_challenge_df["text"])
nb_challenge_probs = nb_model.predict_proba(test_challenge_df["text"])

print("Challenge predictions/probabilities ready.")

Challenge predictions/probabilities ready.


simple disagreement preview

In [ ]:
challenge_pred_df = pd.DataFrame({
    "true_label": test_challenge_df["label"].values,
    "lr_pred": lr_challenge_pred,
    "svm_pred": svm_challenge_pred,
    "nb_pred": nb_challenge_pred,
})

challenge_pred_df["all_agree"] = (
    (challenge_pred_df["lr_pred"] == challenge_pred_df["svm_pred"]) &
    (challenge_pred_df["lr_pred"] == challenge_pred_df["nb_pred"])
).astype(int)

print(challenge_pred_df.head(10))
print("\nAll agree rate on challenge:", challenge_pred_df["all_agree"].mean())

               true_label                 lr_pred                svm_pred  \
0  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
1  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
2  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
3  Appointment Scheduling        Referral Request        Referral Request   
4  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
5  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
6  Appointment Scheduling   Billing and Insurance   Billing and Insurance   
7  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
8  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   
9  Appointment Scheduling  Appointment Scheduling  Appointment Scheduling   

                  nb_pred  all_agree  
0  Appointment Scheduling          1  
1  Appointment Scheduling          1  
2  Appointment Scheduling          

# Richer Shadow Features from Multiple Main Models

helper for probability features

In [47]:
import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

def prob_feature_block(probas, prefix):
    sorted_probs = np.sort(probas, axis=1)
    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2]
    margin = top1 - top2
    ent = np.apply_along_axis(lambda x: scipy_entropy(x + 1e-12), 1, probas)

    return pd.DataFrame({
        f"{prefix}_max_conf": top1,
        f"{prefix}_margin": margin,
        f"{prefix}_entropy": ent,
    })

build richer challenge features first

In [48]:
# Validation / challenge predictions already available for challenge
# Need validation predictions from the 3 models too:
lr_val_pred = lr_model.predict(val_df["text"])
lr_val_probs = lr_model.predict_proba(val_df["text"])

svm_val_pred = svm_model.predict(val_df["text"])
svm_val_probs = svm_model.predict_proba(val_df["text"])

nb_val_pred = nb_model.predict(val_df["text"])
nb_val_probs = nb_model.predict_proba(val_df["text"])

agreement feature helper

In [49]:
def agreement_features(pred_a, pred_b, pred_c):
    pred_df = pd.DataFrame({
        "a": pred_a,
        "b": pred_b,
        "c": pred_c
    })

    num_unique = pred_df.nunique(axis=1)

    all_agree = (num_unique == 1).astype(int)
    any_disagree = (num_unique > 1).astype(int)

    majority_vote_size = []
    for _, row in pred_df.iterrows():
        counts = row.value_counts()
        majority_vote_size.append(counts.iloc[0])

    return pd.DataFrame({
        "all_agree": all_agree,
        "any_disagree": any_disagree,
        "num_unique_preds": num_unique,
        "majority_vote_size": majority_vote_size,
    })

build richer shadow train set from validation

In [50]:
# error target from calibrated LR main model on validation
rich_shadow_train_y = (lr_val_pred != val_df["label"]).astype(int)

rich_shadow_train_X = pd.concat([
    prob_feature_block(lr_val_probs, "lr"),
    prob_feature_block(svm_val_probs, "svm"),
    prob_feature_block(nb_val_probs, "nb"),
    agreement_features(lr_val_pred, svm_val_pred, nb_val_pred),
    pd.DataFrame({
        "text_len_chars": [len(str(t)) for t in val_df["text"]],
        "text_len_words": [len(str(t).split()) for t in val_df["text"]],
    })
], axis=1)

print("Rich shadow train shape:", rich_shadow_train_X.shape)
print(pd.Series(rich_shadow_train_y).value_counts())
print("Validation error rate:", rich_shadow_train_y.mean())

Rich shadow train shape: (180, 15)
label
0    179
1      1
Name: count, dtype: int64
Validation error rate: 0.005555555555555556


build richer challenge test set

In [51]:
rich_shadow_test_y = (lr_challenge_pred != test_challenge_df["label"]).astype(int)

rich_shadow_test_X = pd.concat([
    prob_feature_block(lr_challenge_probs, "lr"),
    prob_feature_block(svm_challenge_probs, "svm"),
    prob_feature_block(nb_challenge_probs, "nb"),
    agreement_features(lr_challenge_pred, svm_challenge_pred, nb_challenge_pred),
    pd.DataFrame({
        "text_len_chars": [len(str(t)) for t in test_challenge_df["text"]],
        "text_len_words": [len(str(t).split()) for t in test_challenge_df["text"]],
    })
], axis=1)

print("Rich shadow test shape:", rich_shadow_test_X.shape)
print(pd.Series(rich_shadow_test_y).value_counts())
print("Challenge error rate:", rich_shadow_test_y.mean())

Rich shadow test shape: (72, 15)
label
0    62
1    10
Name: count, dtype: int64
Challenge error rate: 0.1388888888888889


train richer logistic shadow model

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

rich_shadow_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

rich_shadow_model.fit(rich_shadow_train_X, rich_shadow_train_y)

rich_shadow_risk = rich_shadow_model.predict_proba(rich_shadow_test_X)[:, 1]

print("Rich shadow model trained.")

if len(np.unique(rich_shadow_test_y)) == 2:
    print("Rich shadow ROC-AUC:", roc_auc_score(rich_shadow_test_y, rich_shadow_risk))
    print("Rich shadow AP:", average_precision_score(rich_shadow_test_y, rich_shadow_risk))

Rich shadow model trained.
Rich shadow ROC-AUC: 0.7225806451612904
Rich shadow AP: 0.37044745769121046


inspect coefficients

In [53]:
rich_coef_df = pd.DataFrame({
    "feature": rich_shadow_train_X.columns,
    "coef": rich_shadow_model.coef_[0]
}).sort_values("coef", ascending=False)

print(rich_coef_df)

               feature      coef
11    num_unique_preds  1.237638
10        any_disagree  1.237582
5          svm_entropy  1.229381
8           nb_entropy  0.279996
2           lr_entropy  0.210837
13      text_len_chars  0.013315
14      text_len_words -0.165851
0          lr_max_conf -0.233494
6          nb_max_conf -0.257260
1            lr_margin -0.359723
7            nb_margin -0.362139
3         svm_max_conf -0.664921
4           svm_margin -1.178227
12  majority_vote_size -1.237415
9            all_agree -1.237526


# Corrected Rich Shadow Model (OOF Multi Model Features)

imports

In [54]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

define fresh base models for OOF generation

In [55]:
lr_base_oof = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

svm_base_oof = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

nb_base_oof = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=7000
    )),
    ("clf", MultinomialNB())
])

print("OOF base models defined.")

OOF base models defined.


generate OOF predictions/probabilities for all 3 models

In [56]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train_all = train_df["text"].reset_index(drop=True)
y_train_all = train_df["label"].reset_index(drop=True)

n_train = len(train_df)
n_classes = len(labels_sorted)

# storage
lr_oof_probs = np.zeros((n_train, n_classes))
svm_oof_probs = np.zeros((n_train, n_classes))
nb_oof_probs = np.zeros((n_train, n_classes))

lr_oof_pred = np.empty(n_train, dtype=object)
svm_oof_pred = np.empty(n_train, dtype=object)
nb_oof_pred = np.empty(n_train, dtype=object)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_all, y_train_all), 1):
    X_tr, X_va = X_train_all.iloc[tr_idx], X_train_all.iloc[va_idx]
    y_tr = y_train_all.iloc[tr_idx]

    # LR
    lr_fold = clone(lr_base_oof)
    lr_fold.fit(X_tr, y_tr)
    lr_oof_probs[va_idx] = lr_fold.predict_proba(X_va)
    lr_oof_pred[va_idx] = lr_fold.predict(X_va)

    # NB
    nb_fold = clone(nb_base_oof)
    nb_fold.fit(X_tr, y_tr)
    nb_oof_probs[va_idx] = nb_fold.predict_proba(X_va)
    nb_oof_pred[va_idx] = nb_fold.predict(X_va)

    # SVM + calibration on fold-heldout validation
    svm_fold_base = clone(svm_base_oof)
    svm_fold_base.fit(X_tr, y_tr)

    svm_fold = CalibratedClassifierCV(
        FrozenEstimator(svm_fold_base),
        method="sigmoid"
    )
    svm_fold.fit(X_va, y_train_all.iloc[va_idx])

    svm_oof_probs[va_idx] = svm_fold.predict_proba(X_va)
    svm_oof_pred[va_idx] = svm_fold.predict(X_va)

    print(f"Finished fold {fold}")

print("OOF predictions complete.")

Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
OOF predictions complete.


helper functions for richer features

In [57]:
import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

def prob_feature_block(probas, prefix):
    sorted_probs = np.sort(probas, axis=1)
    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2]
    margin = top1 - top2
    ent = np.apply_along_axis(lambda x: scipy_entropy(x + 1e-12), 1, probas)

    return pd.DataFrame({
        f"{prefix}_max_conf": top1,
        f"{prefix}_margin": margin,
        f"{prefix}_entropy": ent,
    })

def agreement_features(pred_a, pred_b, pred_c):
    pred_df = pd.DataFrame({
        "a": pred_a,
        "b": pred_b,
        "c": pred_c
    })

    num_unique = pred_df.nunique(axis=1)
    all_agree = (num_unique == 1).astype(int)
    any_disagree = (num_unique > 1).astype(int)

    majority_vote_size = []
    for _, row in pred_df.iterrows():
        counts = row.value_counts()
        majority_vote_size.append(counts.iloc[0])

    return pd.DataFrame({
        "all_agree": all_agree,
        "any_disagree": any_disagree,
        "num_unique_preds": num_unique,
        "majority_vote_size": majority_vote_size,
    })

build corrected rich shadow training set

In [58]:
corrected_shadow_train_y = (lr_oof_pred != y_train_all.values).astype(int)

corrected_shadow_train_X = pd.concat([
    prob_feature_block(lr_oof_probs, "lr"),
    prob_feature_block(svm_oof_probs, "svm"),
    prob_feature_block(nb_oof_probs, "nb"),
    agreement_features(lr_oof_pred, svm_oof_pred, nb_oof_pred),
    pd.DataFrame({
        "text_len_chars": [len(str(t)) for t in X_train_all],
        "text_len_words": [len(str(t).split()) for t in X_train_all],
    })
], axis=1)

print("Corrected rich shadow train shape:", corrected_shadow_train_X.shape)
print("\nCorrected shadow target distribution:")
print(pd.Series(corrected_shadow_train_y).value_counts())
print("\nCorrected shadow error rate on train OOF:", corrected_shadow_train_y.mean())

Corrected rich shadow train shape: (360, 15)

Corrected shadow target distribution:
0    339
1     21
Name: count, dtype: int64

Corrected shadow error rate on train OOF: 0.058333333333333334


build corrected rich shadow test set on challenge

In [59]:
# challenge predictions from full models
lr_challenge_pred = lr_model.predict(test_challenge_df["text"])
lr_challenge_probs = lr_model.predict_proba(test_challenge_df["text"])

svm_challenge_pred = svm_model.predict(test_challenge_df["text"])
svm_challenge_probs = svm_model.predict_proba(test_challenge_df["text"])

nb_challenge_pred = nb_model.predict(test_challenge_df["text"])
nb_challenge_probs = nb_model.predict_proba(test_challenge_df["text"])

corrected_shadow_test_y = (lr_challenge_pred != test_challenge_df["label"]).astype(int)

corrected_shadow_test_X = pd.concat([
    prob_feature_block(lr_challenge_probs, "lr"),
    prob_feature_block(svm_challenge_probs, "svm"),
    prob_feature_block(nb_challenge_probs, "nb"),
    agreement_features(lr_challenge_pred, svm_challenge_pred, nb_challenge_pred),
    pd.DataFrame({
        "text_len_chars": [len(str(t)) for t in test_challenge_df["text"]],
        "text_len_words": [len(str(t).split()) for t in test_challenge_df["text"]],
    })
], axis=1)

print("Corrected rich shadow test shape:", corrected_shadow_test_X.shape)
print("\nCorrected challenge target distribution:")
print(pd.Series(corrected_shadow_test_y).value_counts())
print("\nCorrected challenge error rate:", corrected_shadow_test_y.mean())

Corrected rich shadow test shape: (72, 15)

Corrected challenge target distribution:
label
0    62
1    10
Name: count, dtype: int64

Corrected challenge error rate: 0.1388888888888889


train corrected rich shadow model

In [60]:
corrected_rich_shadow_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

corrected_rich_shadow_model.fit(corrected_shadow_train_X, corrected_shadow_train_y)

corrected_rich_shadow_risk = corrected_rich_shadow_model.predict_proba(corrected_shadow_test_X)[:, 1]

print("Corrected rich shadow model trained.")

if len(np.unique(corrected_shadow_test_y)) == 2:
    corrected_roc_auc = roc_auc_score(corrected_shadow_test_y, corrected_rich_shadow_risk)
    corrected_ap = average_precision_score(corrected_shadow_test_y, corrected_rich_shadow_risk)

    print(f"Corrected rich shadow ROC-AUC: {corrected_roc_auc:.4f}")
    print(f"Corrected rich shadow AP: {corrected_ap:.4f}")

Corrected rich shadow model trained.
Corrected rich shadow ROC-AUC: 0.8435
Corrected rich shadow AP: 0.5023


inspect coefficients

In [61]:
corrected_rich_coef_df = pd.DataFrame({
    "feature": corrected_shadow_train_X.columns,
    "coef": corrected_rich_shadow_model.coef_[0]
}).sort_values("coef", ascending=False)

print(corrected_rich_coef_df)

               feature      coef
5          svm_entropy  2.111242
8           nb_entropy  0.616544
11    num_unique_preds  0.458285
10        any_disagree  0.456668
2           lr_entropy  0.225175
14      text_len_words  0.225092
13      text_len_chars -0.043254
0          lr_max_conf -0.206602
1            lr_margin -0.389788
12  majority_vote_size -0.451817
9            all_agree -0.455051
6          nb_max_conf -0.616585
7            nb_margin -0.707255
3         svm_max_conf -1.792517
4           svm_margin -3.325290


abstention table for corrected rich shadow model

In [62]:
corrected_shadow_results = []
for t in risk_thresholds:
    corrected_shadow_results.append(
        selective_metrics_from_risk(
            y_true=test_challenge_df["label"],
            y_pred_labels=lr_challenge_pred,   # main predictions come from LR main model
            risk_scores=corrected_rich_shadow_risk,
            risk_threshold=t
        )
    )

corrected_shadow_results_df = pd.DataFrame(corrected_shadow_results)

print("CORRECTED RICH SHADOW selective prediction results on challenge:")
print(corrected_shadow_results_df)

CORRECTED RICH SHADOW selective prediction results on challenge:
   risk_threshold  coverage  accuracy_on_kept  error_on_kept  num_kept
0             0.2  0.486111          1.000000       0.000000        35
1             0.3  0.541667          1.000000       0.000000        39
2             0.4  0.625000          0.977778       0.022222        45
3             0.5  0.694444          0.940000       0.060000        50
4             0.6  0.750000          0.907407       0.092593        54
5             0.7  0.805556          0.913793       0.086207        58
6             0.8  0.847222          0.901639       0.098361        61


save outputs

In [63]:
corrected_shadow_results_df.to_csv("corrected_rich_shadow_selective_results_challenge.csv", index=False)

corrected_shadow_out = test_challenge_df[["text", "label", "difficulty"]].copy()
corrected_shadow_out["main_pred"] = lr_challenge_pred
corrected_shadow_out["main_correct"] = (corrected_shadow_out["label"] == corrected_shadow_out["main_pred"]).astype(int)
corrected_shadow_out["corrected_rich_shadow_risk"] = corrected_rich_shadow_risk

corrected_shadow_out.to_csv("corrected_rich_shadow_test_challenge_predictions.csv", index=False)

print("Saved:")
print("- corrected_rich_shadow_selective_results_challenge.csv")
print("- corrected_rich_shadow_test_challenge_predictions.csv")

Saved:
- corrected_rich_shadow_selective_results_challenge.csv
- corrected_rich_shadow_test_challenge_predictions.csv
